# From Tickets to a Reliable Summary

[![Open Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Download this notebook, open Colab, and choose **File > Upload notebook**. The full draft course is distributed separately from the public Week 1 repository.


A manager wants the number of active requests in each category. A pipeline can
answer that question, but an innocent-looking array operation can inflate the
counts. We will see the error, explain it, and keep invalid statuses out of new
documents.

This is a fresh four-ticket teaching case, not the complete 12-ticket CSV dataset.
You do not need Week 10's database. The default runs aggregation locally with
`mongomock`. Atlas runs the same pipeline on MongoDB and additionally enforces
the collection validator. The local library does **not** implement that server
feature; its validation section is a clearly labeled trace to interpret.

## Connection or Local Practice

Keep `USE_ATLAS = False` to begin without an account. Package installation still
requires internet access. For Atlas, use your own existing Free cluster and set
the switch to `True`. The next cell prints a new practice database name and this
runtime's public IPv4 address. It contacts ipify without database credentials.

Colab runs on Google's computer, not your laptop. **Add Current IP** in your
laptop's browser can therefore add the wrong address. Use the printed `/32`
address, which allows one IPv4 address. Do not open access from everywhere.

In [ ]:
%pip -q install "pymongo>=4.13,<5" "mongomock>=4.3,<5"

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from ipaddress import IPv4Address
from pprint import pprint
from urllib.request import urlopen
from uuid import uuid4

import mongomock
from pymongo import MongoClient
from pymongo.errors import OperationFailure
from pymongo.server_api import ServerApi

USE_ATLAS = False
database_name = "cst4714_pipeline_" + uuid4().hex[:8]
print("Practice database:", database_name)

if USE_ATLAS:
    try:
        with urlopen("https://api.ipify.org", timeout=10) as response:
            runtime_ip = str(IPv4Address(response.read().decode().strip()))
        print("Temporary Atlas IP access-list entry:", runtime_ip + "/32")
    except Exception:
        raise RuntimeError(
            "The runtime IP check failed. Retry or use local mode; "
            "do not open access from everywhere."
        ) from None

**Atlas only: pause before connecting.** In **Network Access / IP Access List**,
add the printed address as a temporary entry and wait for it to become active.
Recheck after a runtime restart or an outgoing IP change. Your database user is
different from your Atlas website login. It needs read/write access to this
practice database and permission to modify its collection validator (`collMod`).
Ask the instructor to help scope that permission rather than changing a shared
collection or disabling validation. Local mode remains available.

In **Connect > Drivers**, select Python and copy the driver URI. Replace its
password placeholder; reserved characters in the password need URI percent
encoding. Enter the finished URI only in the hidden prompt, never a code cell.
`ping` tests whether MongoDB responds, not whether every later operation is
authorized. Keep TLS verification enabled. Check the runtime IP, database user,
URI, deployment state, and current driver if connection fails.

Run setup once per experiment. Clean up before creating a new practice name.
These handles select a database and collection; the first write stores data.

In [ ]:
client = None
mongodb_uri = None
if USE_ATLAS:
    try:
        mongodb_uri = getpass("Atlas driver URI (hidden): ")
        client = MongoClient(
            mongodb_uri, tls=True, tlsInsecure=False,
            server_api=ServerApi("1", strict=True, deprecation_errors=True),
            serverSelectionTimeoutMS=10000, timeoutMS=10000,
        )
        client.admin.command("ping")
        print("MongoDB responded to ping.")
    except Exception:
        if client is not None:
            client.close()
        raise RuntimeError(
            "Atlas connection failed. Check the runtime IP rule, database "
            "credentials, driver URI, deployment state, and DNS/TLS. "
            "Keep certificate verification enabled; local mode is available."
        ) from None
    finally:
        mongodb_uri = None
else:
    client = mongomock.MongoClient()
    print("Local aggregation practice. Server validation will not execute.")

database = client[database_name]
tickets = database["tickets"]

## 1. Read the Four Tickets Before Querying

Each document is one ticket. `events` is a list of events **inside** that ticket.
An empty list is valid. A resolved ticket still exists but does not belong in an
active workload count. In this course, active means `new`, `open`, or `in_progress`.

Predict the active IDs and the number of active tickets in each category from
the fixture. This gives us a check independent of our aggregation syntax.

In [ ]:
fixture = [
    {"ticket_id": 1001, "category": "streetlight", "status": "open",
     "priority": "urgent", "subject": "Dark streetlight", "assignee_id": 201,
     "opened_at": datetime(2026, 2, 1, tzinfo=timezone.utc),
     "events": [{"type": "created"}, {"type": "assigned"}]},
    {"ticket_id": 1002, "category": "sanitation", "status": "in_progress",
     "priority": "high", "subject": "Missed pickup", "assignee_id": 202,
     "opened_at": datetime(2026, 2, 2, tzinfo=timezone.utc),
     "events": [{"type": "created"}]},
    {"ticket_id": 1003, "category": "streetlight", "status": "resolved",
     "priority": "urgent", "subject": "Lamp repaired", "assignee_id": 201,
     "opened_at": datetime(2026, 2, 3, tzinfo=timezone.utc),
     "events": [{"type": "created"}, {"type": "resolved"}]},
    {"ticket_id": 1004, "category": "streetlight", "status": "new",
     "priority": "low", "subject": "Flickering lamp", "assignee_id": None,
     "opened_at": datetime(2026, 2, 4, tzinfo=timezone.utc), "events": []},
]
# Reset only this notebook's collection when repeating the fixture cell.
tickets.create_index("ticket_id", unique=True)
tickets.delete_many({})
tickets.insert_many(fixture)
pprint(list(tickets.find({}, {"_id": 0, "ticket_id": 1, "category": 1, "status": 1})))

## 2. Build a Summary One Stage at a Time

`$match` selects input documents. `$group` creates one output for each distinct
group key. `$sum: 1` adds one for each input reaching that group, regardless of
how many documents existed before earlier stages.

The dollar sign in `"$category"` means read that field's value. The string
`"category"` without the dollar would group every input under one constant label.
`$project` chooses the final shape, and `$sort` makes display order predictable.

In [ ]:
active_filter = {"status": {"$in": ["new", "open", "in_progress"]}}

pipeline = [
    {"$match": active_filter},  # Input/output: one ticket per document.
    {"$group": {"_id": "$category", "active_count": {"$sum": 1}}},
    {"$project": {"_id": 0, "category": "$_id", "active_count": 1}},
    {"$sort": {"category": 1}},  # Output: one summary per category.
]

for end in range(1, len(pipeline) + 1):
    result = list(tickets.aggregate(pipeline[:end]))
    print("After", list(pipeline[end - 1])[0], ":", len(result), "documents")
    pprint(result)

### Your Change: Urgent Work and Newest Request

Add two outputs to the existing `$group` and expose them in `$project`:

```python
"urgent_count": {"$sum": {"$cond": [{"$eq": ["$priority", "urgent"]}, 1, 0]}},
"newest_opening": {"$max": "$opened_at"}
```

`$cond` chooses 1 or 0 per input ticket. `$max` keeps the largest date. Explain why
the resolved urgent ticket must not contribute to this **active** summary. Run
the modified pipeline and check streetlight by looking back at its ticket IDs.
Keep the active-count result when you add the two new fields.

In [ ]:
# Edit these stages with the two expressions above, then rerun this cell.
my_pipeline = [
    {"$match": active_filter},
    {"$group": {"_id": "$category", "active_count": {"$sum": 1}}},
    {"$project": {"_id": 0, "category": "$_id", "active_count": 1}},
    {"$sort": {"category": 1}},
]
my_result = list(tickets.aggregate(my_pipeline))
pprint(my_result)

# An independent count checks the original report, not the new fields yet.
streetlight_count = tickets.count_documents({**active_filter, "category": "streetlight"})
print("Independent active streetlight count:", streetlight_count)
assert streetlight_count == 2

## 3. Discover a Counting Trap

Suppose we insert `$unwind: "$events"` before grouping. It produces one document
per array element. A ticket with two events now contributes twice; a ticket with
an empty array disappears unless we request preservation.

Predict the ticket IDs in the output. The total number of documents can
accidentally match the ticket count even when the report is wrong.

In [ ]:
event_rows = list(tickets.aggregate([
    {"$match": active_filter},
    {"$unwind": "$events"},
    {"$project": {"_id": 0, "ticket_id": 1, "event_type": "$events.type"}},
    {"$sort": {"ticket_id": 1, "event_type": 1}},
]))
pprint(event_rows)
print("Active ticket IDs:", [row["ticket_id"] for row in tickets.find(active_filter)])
print("Unwound row IDs:", [row["ticket_id"] for row in event_rows])

For a ticket count, group the tickets without unwinding. For an **event** count,
unwinding is appropriate, but label the report as events. Preserving empty arrays
does not undo duplication for tickets with multiple events. SQL joins can cause
the same change of grain; remember the ticket-to-event relationship from Week 2.

## 4. Require Valid Statuses Without Requiring Every Field

A validator applies to writes. It does not summarize data or make a query faster.
This schema requires a numeric ticket ID, an allowed status, and a subject.
Other fields may exist without being required. That is controlled flexibility.

The validator below is a MongoDB command document. It is not Python's type system
and is not identical to standard JSON Schema. BSON adds types such as `date`.

In [ ]:
validator = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["ticket_id", "status", "subject"],
        "properties": {
            "ticket_id": {"bsonType": ["int", "long"]},
            "status": {"enum": ["new", "open", "in_progress", "resolved", "closed"]},
            "subject": {"bsonType": "string"},
        },
    }
}

if USE_ATLAS:
    database.command({"collMod": "tickets", "validator": validator,
                      "validationLevel": "strict", "validationAction": "error"})
    print("MongoDB now checks new inserts and updates.")
else:
    print("Validator defined but NOT installed: mongomock has no server validation.")

### Your Change: Opening Date

Require `opened_at`, and give it `{"bsonType": "date"}` in `properties`. Edit the
validator cell and rerun it. All four baseline tickets already have BSON dates.

The next test cell includes an allowed date, a date written as text, and a
missing date. A string that looks like a date is not a BSON date. The text and
missing cases should be rejected only after you add and reinstall your date rule.

Installing validation does not retroactively delete or repair existing invalid
documents. Inspect existing records before tightening a live collection. This
fresh practice fixture avoids an accidental migration of somebody else's data.

In [ ]:
valid_test = {"ticket_id": 1099, "status": "new", "subject": "Validation test",
              "opened_at": datetime(2026, 2, 5, tzinfo=timezone.utc)}
invalid_tests = [
    ("Invalid status", {**valid_test, "ticket_id": 1100, "status": "almost_done"}),
]
date_rule_added = (
    "opened_at" in validator["$jsonSchema"]["required"]
    and validator["$jsonSchema"]["properties"].get("opened_at") == {"bsonType": "date"}
)
if date_rule_added:
    invalid_tests += [
        ("Date stored as text", {**valid_test, "ticket_id": 1101,
                                "opened_at": "2026-02-05T00:00:00Z"}),
        ("Missing date", {"ticket_id": 1102, "status": "new", "subject": "No date"}),
    ]
else:
    print("Date rule not enabled yet. Edit and rerun the validator cell, then this test.")

if USE_ATLAS:
    try:
        # These IDs belong only to this test, so repeating it starts cleanly.
        tickets.delete_many({"ticket_id": {"$in": [1099, 1100, 1101, 1102]}})
        tickets.insert_one(valid_test)
        print("Valid insert accepted:", tickets.count_documents({"ticket_id": 1099}))
        for label, document in invalid_tests:
            try:
                tickets.insert_one(document)
            except OperationFailure as error:
                if error.code != 121:
                    raise
                print(label + " rejected. MongoDB error code:", error.code)
            else:
                raise AssertionError(label + " was accepted. Inspect the installed validator.")
    finally:
        tickets.delete_many({"ticket_id": {"$in": [1099, 1100, 1101, 1102]}})
else:
    print("Trace to interpret, NOT a result from this runtime:")
    print("Valid insert accepted: 1")
    for label, document in invalid_tests:
        print(label + " rejected. MongoDB error code: 121")

## Explain Your Result

In this one cell, write a short response:

- Give your streetlight summary, including urgent count and newest opening.
  Identify the source tickets that support it.
- Explain why the unwound rows are unsuitable for counting requests, even though
  their total equals the number of active tickets in this fixture.
- Describe your opening-date rule. Name whether you executed MongoDB validation
  or interpreted the supplied trace. Explain both the text-date and missing-date
  outcomes, and one rule the validator still does not
  enforce (for example, whether an assignee ID exists in another collection).

Optional extension: add a set of non-null assignee IDs or count events by type.
These are extensions, not more required deliverables.

## Your Explanation

Replace this paragraph with your response. Keep it in this notebook; no separate
report is needed.

## Cleanup

The next cell deletes only `tickets` in the uniquely named practice database.
Run it when finished and remove the temporary Atlas IP entry in the dashboard.
A closed Python connection does not stop or delete an Atlas cluster. Do not delete
a shared cluster or another project's data. Do not put project data in this
practice collection.

In [ ]:
database.drop_collection("tickets")
client.close()
print("Removed this run's practice collection and closed its client.")

## Further Reading

- [MongoDB aggregation pipeline](https://www.mongodb.com/docs/manual/core/aggregation-pipeline/)
- [Unwind behavior](https://www.mongodb.com/docs/manual/reference/operator/aggregation/unwind/)
- [Schema validation](https://www.mongodb.com/docs/manual/core/schema-validation/)
- [Atlas IP access list](https://www.mongodb.com/docs/atlas/security/ip-access-list/)

Course prose: CC BY-NC-SA 4.0. Code: MIT. Synthetic fixture: CC0.